# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [41]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor,AdaBoostRegressor, GradientBoostingRegressor

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

from xgboost import XGBRegressor

In [42]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [43]:
spaceship.dropna(inplace=True) 

In [44]:
spaceship['Cabin'] = spaceship['Cabin'].str.partition('/')[0]

In [45]:
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [46]:
spaceship = spaceship.drop(columns=['PassengerId', 'Name'])

In [47]:
cols = ['CryoSleep', 'VIP']
spaceship[cols] = spaceship[cols].astype(int)
cols = ['HomePlanet', 'Destination', 'Cabin']

for col in cols:
    dummies = pd.get_dummies(
        spaceship[col],
        prefix=col
    )

    spaceship = pd.concat([spaceship, dummies], axis=1)

spaceship = spaceship.drop(columns=['HomePlanet', 'Destination', 'Cabin'])


**Perform Train Test Split**

In [48]:
# Features and target
features = spaceship.drop(columns=["Transported"])
target = spaceship["Transported"]

In [49]:
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size = 0.20, random_state=0)

In [50]:
scaler = StandardScaler()

X_train_norm = scaler.fit_transform(X_train)
X_test_norm = scaler.transform(X_test)

In [60]:
X_train_norm = pd.DataFrame(X_train_norm,columns=X_train.columns)

X_test_norm = pd.DataFrame(X_test_norm,columns=X_test.columns)

**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [61]:
bagging_reg = BaggingRegressor(DecisionTreeRegressor(max_depth=20),
                               n_estimators=100,
                               max_samples = 1000)

In [62]:
bagging_reg.fit(X_train_norm, y_train)

BaggingRegressor(estimator=DecisionTreeRegressor(max_depth=20),
                 max_samples=1000, n_estimators=100)

In [63]:
pred = bagging_reg.predict(X_test_norm)

print("MAE", mean_absolute_error(y_test, pred))

rmse = np.sqrt(mean_squared_error(y_test, pred))
print("RMSE", rmse)

print("R2 score", bagging_reg.score(X_test_norm, y_test))

MAE 0.2766564428529155
RMSE 0.3752722282669433
R2 score 0.43668301876625293


- Random Forests

In [64]:
forest = RandomForestRegressor(n_estimators=100,
                             max_depth=20)

In [65]:
forest.fit(X_train_norm, y_train)

RandomForestRegressor(max_depth=20)

In [67]:
pred = forest.predict(X_test_norm)

print("MAE", mean_absolute_error(y_test, pred))

rmse = np.sqrt(mean_squared_error(y_test, pred))
print("RMSE", rmse)

print("R2 score", forest.score(X_test_norm, y_test))

MAE 0.2695602543794622
RMSE 0.38336490510365656
R2 score 0.4121253981394576


- Gradient Boosting

In [72]:
gb_reg = GradientBoostingRegressor(max_depth=20,
                                   n_estimators=100)

In [73]:
gb_reg.fit(X_train_norm, y_train)

GradientBoostingRegressor(max_depth=20)

In [74]:
pred = gb_reg.predict(X_test_norm)

print("MAE", mean_absolute_error(y_test, pred))

rmse = np.sqrt(mean_squared_error(y_test, pred))
print("RMSE", rmse)

print("R2 score", gb_reg.score(X_test_norm, y_test))

MAE 0.26171674990967597
RMSE 0.4242134366315484
R2 score 0.280171840725005


- Adaptive Boosting

In [68]:
ada_reg = AdaBoostRegressor(DecisionTreeRegressor(max_depth=20),
                            n_estimators=100)

In [70]:
ada_reg.fit(X_train_norm, y_train)

AdaBoostRegressor(estimator=DecisionTreeRegressor(max_depth=20),
                  n_estimators=100)

In [71]:
pred = ada_reg.predict(X_test_norm)

print("MAE", mean_absolute_error(y_test, pred))

rmse = np.sqrt(mean_squared_error(y_test, pred))
print("RMSE", rmse)

print("R2 score", ada_reg.score(X_test_norm, y_test))

MAE 0.25168658426147056
RMSE 0.43700012603879657
R2 score 0.23612355936830365


## Comparing models in a loop

In [76]:
datasets = [
    {"name": "raw",        "X_train": X_train,      "X_test": X_test},
    {"name": "normalized", "X_train": X_train_norm, "X_test": X_test_norm}
]

In [77]:
models = [
    {"name": "Bagging",          "model": BaggingRegressor(DecisionTreeRegressor(max_depth=20), n_estimators=100, max_samples=1000)},
    {"name": "RandomForest",     "model": RandomForestRegressor(n_estimators=100, max_depth=20)},
    {"name": "AdaBoost",         "model": AdaBoostRegressor(DecisionTreeRegressor(max_depth=20), n_estimators=100)},
    {"name": "GradientBoosting", "model": GradientBoostingRegressor(n_estimators=100, max_depth=20)},
    {"name": "XGBoost",          "model": XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1)}]

In [78]:
results = []

for ds in datasets:
    for m in models:
        m["model"].fit(ds["X_train"], y_train)
        pred = m["model"].predict(ds["X_test"])

        results.append({
            "dataset": ds["name"],
            "model":   m["name"],
            "MAE":     mean_absolute_error(y_test, pred),
            "RMSE":    np.sqrt(mean_squared_error(y_test, pred)),
            "R2":      r2_score(y_test, pred),
        })
        print(f"{ds['name']:>10} | {m['name']:<16} done")

results_df = pd.DataFrame(results).sort_values("R2", ascending=False).reset_index(drop=True)
results_df

       raw | Bagging          done
       raw | RandomForest     done
       raw | AdaBoost         done
       raw | GradientBoosting done
       raw | XGBoost          done
normalized | Bagging          done
normalized | RandomForest     done
normalized | AdaBoost         done
normalized | GradientBoosting done
normalized | XGBoost          done


,dataset,model,MAE,RMSE,R2
0,normalized,Bagging,0.277608,0.377050,0.431334
1,raw,Bagging,0.278794,0.378770,0.426134
2,normalized,XGBoost,0.273401,0.378835,0.425935
3,raw,XGBoost,0.273401,0.378835,0.425935
4,raw,RandomForest,0.267820,0.382871,0.413639
5,normalized,RandomForest,0.267971,0.383953,0.410320
6,normalized,AdaBoost,0.242130,0.414783,0.311820
7,raw,GradientBoosting,0.261440,0.420700,0.292048
8,normalized,GradientBoosting,0.264665,0.421976,0.287744
9,raw,AdaBoost,0.249432,0.430026,0.260311


Which model is the best and why?

The best model is Bagging with 0.4313 R2 and MAE 0.2776, RMSE 0.377050. The R2 is lower, feature selection should be improve in order to increase the R2